# Projeto Final — RideSmart
## Modelagem e Análise de Rotas Urbanas com Grafos


### Resumo do problema

Dado um ponto de origem `A`, um destino `B` e uma distância máxima de caminhada `X`, o objetivo é
escolher um **ponto de embarque** `P` (a no máximo `X` metros de caminhada de `A`) que minimize o
custo total da viagem, composta por dois trechos:

```text
A → P   (a pé)
P → B   (de carro)
```

O notebook modela a malha viária real como grafo, implementa e compara quatro algoritmos de caminho
mínimo, introduz **trânsito sintético** e analisa o *trade-off* entre caminhar um pouco e chegar
mais rápido.

## 1. Modelagem do problema como grafo

**Grafo dirigido com multiarestas** $G = (V, E)$ obtido do OpenStreetMap via OSMnx:

- **Nós ($V$):** cruzamentos e extremidades de vias. Cada nó possui coordenadas geográficas
  (`y` = latitude, `x` = longitude).
- **Arestas ($E$):** segmentos de rua entre dois cruzamentos. O grafo é um `MultiDiGraph`:
  é **dirigido** (modela ruas de mão única) e admite **arestas paralelas** (duas ruas distintas
  ligando o mesmo par de cruzamentos).

**Funções de custo (pesos das arestas):**

| Peso | Significado | Usado para |
|---|---|---|
| `length` | comprimento do segmento em metros | menor **distância** e caminhada `A→P` |
| `travel_time` | tempo de percurso de carro em segundos (sem trânsito) | rota mais **rápida** |
| `travel_time_traffic` | `travel_time` ajustado por um fator de congestionamento | rota mais rápida **com trânsito** |

O tempo de caminhada é derivado do comprimento por uma velocidade fixa de pedestre
($v_{pe} \approx 1{,}4$ m/s $\approx 5$ km/h):
$$t_{pe}(A \to P) = \frac{\text{distância}_{length}(A \to P)}{v_{pe}}.$$

O **custo total** de uma escolha de embarque `P` é:
$$\text{custo}(P) = t_{pe}(A \to P) + \text{custo}_{carro}(P \to B),$$
sujeito à restrição $\text{distância}_{length}(A \to P) \le X$.

## 2. Configuração do ambiente

Em ambientes como o Google Colab, descomente a instalação abaixo. O OSMnx requer acesso à internet
para baixar os dados do OpenStreetMap.

In [ ]:
!pip install osmnx networkx pandas matplotlib scikit-learn

import time, copy, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import osmnx as ox

ox.settings.use_cache = True          # cacheia o download (re-execuções ficam instantâneas)
ox.settings.log_console = False
print("OSMnx", ox.__version__, "| NetworkX", nx.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 1.5 MB/s eta 0:00:00
OSMnx 2.1.0 | NetworkX 3.6.1


## 3. Download da malha viária real

Baixamos a rede **dirigível** (`network_type="drive"`) num raio em torno de um ponto central da
cidade. Em seguida o OSMnx imputa velocidades por tipo de via (`add_edge_speeds`) e calcula o tempo
de percurso de cada aresta (`add_edge_travel_times`).

In [ ]:
# Centro da área de estudo (exemplo: bairro de Petrópolis / Tirol, Natal-RN).
# Troque pelas coordenadas da sua cidade/bairro.
CENTRO = (-5.8000, -35.2050)   # (lat, lon)
RAIO_M = 2500                  # raio da malha em metros

G = ox.graph_from_point(CENTRO, dist=RAIO_M, network_type="drive")

# velocidades (km/h) e tempos de percurso (s) por aresta
try:
    G = ox.routing.add_edge_speeds(G)
    G = ox.routing.add_edge_travel_times(G)
except AttributeError:                 # compatibilidade com versões antigas
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)

print(f"[drive] Nós: {G.number_of_nodes()}  |  Arestas: {G.number_of_edges()}")
u, v, k, d = list(G.edges(keys=True, data=True))[0]
print(f"  Aresta exemplo: {d.get('name')} | length={d.get('length'):.1f}m | travel_time={d.get('travel_time'):.1f}s")

# Grafo separado para caminhada (inclui calçadas e vias de pedestre)
G_walk = ox.graph_from_point(CENTRO, dist=RAIO_M, network_type="walk")
print(f"[walk]  Nós: {G_walk.number_of_nodes()}  |  Arestas: {G_walk.number_of_edges()}")

[drive] Nós: 3104  |  Arestas: 6762
  Aresta exemplo: Avenida Governador Sílvio Pedroza | length=28.2m | travel_time=1.9s
[walk]  Nós: 4932  |  Arestas: 14462


## 4. Definição de origem `A`, destino `B`, distância máxima `X`

Escolhemos coordenadas reais de `A` e `B` e mapeamos cada uma para o nó mais próximo da malha
(`nearest_nodes`). `X` é a distância máxima que o usuário aceita caminhar; `V_PE` é a velocidade de
caminhada.

In [ ]:
# Coordenadas de origem e destino (lat, lon) — ajuste conforme sua cidade
A_latlon = (-5.7935, -35.2010)
B_latlon = (-5.8120, -35.1980)

X      = 600.0    # distância máxima de caminhada (metros)
V_PE   = 1.4      # velocidade de caminhada (m/s) ~ 5 km/h

A = ox.distance.nearest_nodes(G, A_latlon[1], A_latlon[0])
B = ox.distance.nearest_nodes(G, B_latlon[1], B_latlon[0])
A_walk = ox.distance.nearest_nodes(G_walk, A_latlon[1], A_latlon[0])
print(f"Nó de origem  A = {A}  (walk: {A_walk})")
print(f"Nó de destino B = {B}")

# velocidade máxima do grafo (m/s) — usada na heurística do A* para tempo
VMAX_MS = max(d.get('speed_kph', 1) for _,_,d in G.edges(data=True)) / 3.6
print(f"Velocidade máxima do grafo: {VMAX_MS*3.6:.0f} km/h")

Nó de origem  A = 8626298111  (walk: 7616423477)
Nó de destino B = 503422480
Velocidade máxima do grafo: 70 km/h
